# 🧠 Fetal Ultrasound: Error Analysis & Explainability

This notebook demonstrates how to evaluate the trained model, analyze its failures, and visualize its decision-making process using Grad-CAM.

## 1. Setup and Loading

In [1]:
import sys
import os

# --- Configuration pour Google Colab ---
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    if not os.path.exists('src'):
        print("Environnement Colab détecté. Clonage du repo...")
        !git clone https://github.com/Abd2k27/fetal-ultrasound-classification.git
        %cd fetal-ultrasound-classification
        !pip install -r requirements.txt --quiet
        sys.path.append(os.getcwd())
else:
    # Configuration pour usage local
    if not os.path.exists('src') and os.path.exists('../src'):
        sys.path.insert(0, os.path.abspath('..'))

import torch
from torch.utils.data import DataLoader
from src.dataset import FetalUltrasoundDataset, get_transforms
from src.model import create_fetal_model
from src.evaluation import get_predictions, plot_confusion_matrix, find_worst_predictions, run_gradcam, visualize_gradcam
from src.utils import get_device

device = get_device()

# Détection dynamique des chemins de données
if IS_COLAB:
    # Sur Colab, on s'attend à être à la racine du projet
    data_dir = "data/Images"
    csv_path = "data/FETAL_PLANES_DB_data.csv"
    model_path = "best_model.pth"
    
    if not os.path.exists("data"):
        print("❌ ERREUR: Dossier 'data/' introuvable sur Colab.")
        print("👉 Pensez à uploader et dézipper votre 'data.zip' !")
else:
    # En local, on vérifie si on est dans le dossier notebooks/
    if os.path.exists("../data"):
        data_dir = "../data/Images"
        csv_path = "../data/FETAL_PLANES_DB_data.csv"
        model_path = "../best_model.pth"
    else:
        data_dir = "data/Images"
        csv_path = "data/FETAL_PLANES_DB_data.csv"
        model_path = "best_model.pth"

print(f"Mode: {'Colab' if IS_COLAB else 'Local'}")
print(f"Utilisation des données depuis : {data_dir}")

# Load Test Dataset
test_dataset = FetalUltrasoundDataset(csv_path, data_dir, split='Test', transform=get_transforms(is_train=False))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load Model
model = create_fetal_model(num_classes=len(test_dataset.classes))
checkpoint = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval();

Mode: Local
Utilisation des données depuis : ../data/Images


## 2. Global Performance: Confusion Matrix

In [2]:
y_pred, y_true, y_probs = get_predictions(model, test_loader, device)
plot_confusion_matrix(y_true, y_pred, test_dataset.classes)

## 3. Worst Predictions Analysis
Finding cases where the model was very confident but wrong. These are the most interesting cases for clinical review.

In [3]:
worst = find_worst_predictions(y_true, y_pred, y_probs, test_dataset.classes)
for error in worst:
    print(f"Index: {error['index']} | True: {error['true']} | Predicted: {error['pred']} (Conf: {error['conf']:.2f})")

Index: 963 | True: Other | Predicted: Fetal brain (Conf: 1.00)
Index: 5262 | True: Other | Predicted: Fetal brain (Conf: 1.00)
Index: 679 | True: Other | Predicted: Maternal cervix (Conf: 1.00)
Index: 3504 | True: Other | Predicted: Fetal abdomen (Conf: 1.00)
Index: 622 | True: Other | Predicted: Fetal abdomen (Conf: 1.00)


## 4. Explainability with Grad-CAM
Visualizing the regions of the ultrasound image that contributed most to the model's prediction.

In [4]:
sample_idx = worst[0]['index']
img_tensor, label = test_dataset[sample_idx]
img_tensor = img_tensor.unsqueeze(0).to(device)

# EfficientNet target layer is usually the last block
target_layer = model.conv_head

grayscale_cam = run_gradcam(model, target_layer, img_tensor)
visualize_gradcam(img_tensor, grayscale_cam)